# 02 - Baseline inference

Objectif : tester et documenter la baseline existante sans recréer une baseline différente si `toy_predict` existe.

Fichiers concernés : `src/inference.py`, `src/guardrails.py`, `data/synthetic_cases.csv`, `outputs/baseline_predictions.csv`.

## Ce qu'est la baseline

La baseline actuelle est `src.inference.toy_predict`. Elle est volontairement
déterministe: pour les images synthétiques fournies, elle déduit la classe à partir
du nom du fichier. Cela la rend reproductible et très utile pour tester la chaîne
logicielle: lecture du dataset, génération JSON, garde-fous, exports et métriques.

Mais cette baseline ne prouve aucune capacité médicale. Elle ne démontre pas qu'un
modèle comprend une radiographie. Elle valide seulement que le dépôt sait produire
une sortie structurée dans un cadre jouet.

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "src").exists() and PROJECT_ROOT.parent != PROJECT_ROOT:
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data"
SAMPLE_IMAGES_DIR = DATA_DIR / "sample_images"
SRC_DIR = PROJECT_ROOT / "src"
API_DIR = PROJECT_ROOT / "api"
APP_DIR = PROJECT_ROOT / "app"
EVAL_DIR = PROJECT_ROOT / "eval"
OUTPUTS_DIR = PROJECT_ROOT / "outputs"
PROMPTS_DIR = PROJECT_ROOT / "prompts"
TESTS_DIR = PROJECT_ROOT / "tests"
OUTPUTS_DIR.mkdir(exist_ok=True)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

print("PROJECT_ROOT =", PROJECT_ROOT)

PROJECT_ROOT = c:\Users\Sarah Efrei\OneDrive\Desktop\2025-2026\S6\MasterCamp\Code du prof\ARVI-RX


In [2]:
import pandas as pd
import time
from pathlib import Path
from src.guardrails import apply_safety_guardrails, validate_prediction

try:
    from src.inference import toy_predict
    baseline_fn = toy_predict
    print("Baseline existante détectée: toy_predict")
except Exception:
    def baseline_fn(image_path, mode="baseline"):
        name = Path(image_path).name.lower()
        klass = "suspected_opacity" if "suspected_opacity" in name else "normal" if "normal" in name else "uncertain"
        return {"image_quality": "limited" if klass == "uncertain" else "good", "predicted_class": klass, "confidence": 0.7, "visual_evidence": [], "justification": "Fallback déterministe.", "limitations": ["synthetic toy image"], "warning": "Prototype pédagogique. Non destiné au diagnostic. Validation par un professionnel qualifié requise."}

df = pd.read_csv(DATA_DIR / "synthetic_cases.csv")
image_path = PROJECT_ROOT / df.loc[1, "image_path"]
pred = apply_safety_guardrails(baseline_fn(image_path, mode="baseline"))
print(pred)
print("Contrat OK:", {"predicted_class", "confidence", "warning", "limitations"} <= set(pred), "visual key:", "visual_observations" in pred or "visual_evidence" in pred)

Baseline existante détectée: toy_predict
{'image_quality': 'good', 'predicted_class': 'suspected_opacity', 'confidence': 0.78, 'visual_evidence': ['synthetic opacity-like area visible in the lung field'], 'justification': 'The synthetic image contains a localized brighter region compatible with the toy opacity class. This is a pipeline validation result, not a medical interpretation.', 'limitations': ['synthetic toy image', 'no clinical context', 'not a validated medical model'], 'warning': 'Prototype pédagogique. Non destiné au diagnostic. Validation par un professionnel qualifié requise.', 'model_name': 'toy-rule-baseline', 'prompt_version': 'baseline_v1', 'latency_ms': 0, 'guardrail_errors': []}
Contrat OK: True visual key: True


## Interprétation des résultats de baseline

Si l'accuracy vaut `1.0`, c'est attendu dans ce dépôt: les noms de fichiers
contiennent les indices de classe, et la baseline exploite indirectement ce signal.
C'est un label leakage assumé pour un smoke test pédagogique.

Conclusion réelle:
- oui, la baseline est reproductible;
- oui, le contrat JSON et le warning peuvent être testés;
- non, le score ne mesure pas une performance médicale;
- non, le score ne doit pas être présenté comme une validation clinique.

In [3]:
rows = []
for _, row in df.iterrows():
    image_path = PROJECT_ROOT / row["image_path"]
    start = time.perf_counter()
    pred = apply_safety_guardrails(baseline_fn(image_path, mode="baseline"))
    measured_latency_ms = round((time.perf_counter() - start) * 1000, 3)
    valid, errors = validate_prediction(pred)
    rows.append({
        "case_id": row["case_id"],
        "filename": Path(row["image_path"]).name,
        "image_path": row["image_path"],
        "expected_label": row["label"],
        "predicted_class": pred.get("predicted_class"),
        "confidence": pred.get("confidence"),
        "json_valid": valid,
        "warning_present": bool(pred.get("warning")),
        "latency_ms": pred.get("latency_ms", measured_latency_ms),
        "guardrail_errors": ";".join(errors),
    })
baseline_df = pd.DataFrame(rows)
out = OUTPUTS_DIR / "baseline_predictions.csv"
baseline_df.to_csv(out, index=False)
print(out)
display(baseline_df.head())

c:\Users\Sarah Efrei\OneDrive\Desktop\2025-2026\S6\MasterCamp\Code du prof\ARVI-RX\outputs\baseline_predictions.csv


,case_id,filename,image_path,expected_label,predicted_class,confidence,json_valid,warning_present,latency_ms,guardrail_errors
0,CXR_SYN_001,CXR_SYN_001_normal.png,data/sample_images/CXR_SYN_001_normal.png,normal,normal,0.72,True,True,0,
1,CXR_SYN_002,CXR_SYN_002_suspected_opacity.png,data/sample_images/CXR_SYN_002_suspected_opaci...,suspected_opacity,suspected_opacity,0.78,True,True,0,
2,CXR_SYN_003,CXR_SYN_003_uncertain.png,data/sample_images/CXR_SYN_003_uncertain.png,uncertain,uncertain,0.52,True,True,0,
3,CXR_SYN_004,CXR_SYN_004_normal.png,data/sample_images/CXR_SYN_004_normal.png,normal,normal,0.72,True,True,0,
4,CXR_SYN_005,CXR_SYN_005_suspected_opacity.png,data/sample_images/CXR_SYN_005_suspected_opaci...,suspected_opacity,suspected_opacity,0.78,True,True,0,


Conclusion : `toy_predict` est la baseline reproductible du projet. Elle doit rester
clairement décrite comme une baseline jouet avec label leakage par nom de fichier.
Le contrat final retient la clé officielle `visual_evidence`.